# Lab 6 — Grocery Inventory Analytics (SOLUTION NOTEBOOK)

> ⚠️ **Facilitator reference only** — do not share with participants before the lab.

Completed working code for all 4 stages including CSV joins and SQLite database work.

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import json
import sqlite3
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

DATA_DIR = Path('..') / 'data'

print('✅ Imports OK')

---
## Stage 1 — Data Onboarding & Quality Check

In [ ]:
# Load inventory data
df = pd.read_excel(
    DATA_DIR / 'grocery_inventory.xlsx',
    sheet_name='Inventory',
    parse_dates=['week'],
    dtype={'on_promotion': bool}
)

print(f'Shape: {df.shape}')
print(f'\nDtypes:')
print(df.dtypes)
print(f'\nSample:')
df.head(3)

**What to look for:** 3,000 rows, 12 columns. `week` as datetime64, `on_promotion` as bool.

In [ ]:
# Data quality report
def data_quality_report(df: pd.DataFrame) -> None:
    print('=' * 60)
    print('DATA QUALITY REPORT — grocery_inventory.xlsx')
    print('=' * 60)
    neg = df[df['stock_level'] < 0]
    print(f'\n[A] Negative stock_level: {len(neg)} rows')
    if len(neg):
        print(neg[['sku', 'store_id', 'week', 'stock_level']].to_string(index=False))
    below = df[df['stock_level'] < df['reorder_threshold']]
    print(f'\n[B] Stock below reorder threshold: {len(below)} rows')
    print(below.groupby('category').size().to_string())
    promo_zero = df[(df['on_promotion'] == True) & (df['units_sold'] == 0)]
    print(f'\n[C] Promo active, zero units sold: {len(promo_zero)} rows')
    if len(promo_zero):
        print(promo_zero[['sku', 'product_name', 'store_id', 'week']].to_string(index=False))
    missing_price = df[df['sell_price_eur'].isna()]
    print(f'\n[D] Missing sell_price_eur: {len(missing_price)} rows')
    if len(missing_price):
        print(missing_price[['sku', 'product_name', 'store_id']].to_string(index=False))
    print('\n' + '=' * 60)


data_quality_report(df)

---
## Stage 2 — EDA & External Data Join

In [ ]:
# 1. Sales by category
cat_sales = df.groupby('category')['units_sold'].sum().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
cat_sales.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Total Units Sold (all stores, all weeks)')
ax.set_title('Total Sales Volume by Category', fontsize=13, fontweight='bold')
for i, v in enumerate(cat_sales):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Weekly Beer/Spirits trend
beer_weekly = (
    df[df['category'] == 'Beer/Spirits']
    .groupby('week')['units_sold'].sum().sort_index()
)
fig, ax = plt.subplots(figsize=(12, 4))
beer_weekly.plot(kind='line', ax=ax, marker='o', color='darkorange', linewidth=2)
ax.set_xlabel('Week')
ax.set_ylabel('Total Units Sold')
ax.set_title('Beer/Spirits — Weekly Sales Trend (All Stores)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Store stockout rate
latest_week = df['week'].max()
latest = df[df['week'] == latest_week].copy()
stockout_rate = (
    latest.groupby('store_id')
    .apply(lambda g: (g['stock_level'] < g['reorder_threshold']).sum() / len(g) * 100)
    .sort_values(ascending=False)
    .rename('stockout_rate_pct')
)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if s == stockout_rate.idxmax() else 'steelblue' for s in stockout_rate.index]
stockout_rate.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Store')
ax.set_ylabel('% SKUs Below Reorder Threshold')
ax.set_title(f'Stockout Rate by Store — Week of {latest_week.date()}', fontsize=13, fontweight='bold')
for i, v in enumerate(stockout_rate):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(f'\n⚠️  Highest stockout rate: {stockout_rate.idxmax()} at {stockout_rate.max():.1f}%')

**What to look for:** STR-C highlighted in red — it has a systematic demand/supply imbalance.
This is the visual centrepiece of Stage 2.

In [ ]:
# Load and join external CSVs
supplier_df = pd.read_csv(DATA_DIR / 'supplier_prices.csv')
promo_df    = pd.read_csv(DATA_DIR / 'promo_calendar.csv', parse_dates=['week'])

print(f'supplier_prices: {supplier_df.shape}  — columns: {supplier_df.columns.tolist()}')
print(f'promo_calendar:  {promo_df.shape}  — columns: {promo_df.columns.tolist()}')

# Join onto inventory (SKU-level for supplier, SKU+week for promo)
df_enriched = df.merge(
    supplier_df[['sku', 'supplier_name', 'contract_price_eur', 'price_mismatch', 'min_order_qty']],
    on='sku', how='left'
)
df_enriched = df_enriched.merge(
    promo_df[['sku', 'week', 'promo_type', 'discount_pct']],
    on=['sku', 'week'], how='left'
)

print(f'\nEnriched shape: {df_enriched.shape}')
df_enriched.head(3)

**What to look for:** The join added `supplier_name`, `contract_price_eur`, `price_mismatch`,
`promo_type`, and `discount_pct`. Rows without a promo entry show NaN in promo columns — that's correct.

In [ ]:
# Supplier price mismatch analysis
mismatches = supplier_df[supplier_df['price_mismatch'] == True][
    ['sku', 'product_name', 'supplier_name', 'contract_price_eur', 'inventory_cost_eur']
].copy()
mismatches['gap_eur'] = (mismatches['contract_price_eur'] - mismatches['inventory_cost_eur']).round(2)
mismatches['gap_pct'] = ((mismatches['gap_eur'] / mismatches['inventory_cost_eur']) * 100).round(1)
mismatches = mismatches.sort_values('gap_eur', ascending=False)

print(f'SKUs where supplier has raised price vs inventory records: {len(mismatches)}')
print()
print(mismatches.to_string(index=False))

**What to look for:** These are SKUs where the supplier's current contract price is higher than
what's recorded in the inventory system. The gap in EUR is the per-unit cost underestimation.
Multiply by units_to_order to see total reorder cost exposure.

In [ ]:
# Enriched margin analysis (using contract price and promo discount)
df_enriched['effective_sell_price'] = df_enriched['sell_price_eur'] * (
    1 - df_enriched['discount_pct'].fillna(0) / 100
)
df_enriched['effective_margin_eur'] = (
    df_enriched['effective_sell_price'] - df_enriched['contract_price_eur']
)
df_enriched['effective_margin_pct'] = (
    df_enriched['effective_margin_eur'] / df_enriched['effective_sell_price'] * 100
).round(1)

cat_margin = (
    df_enriched.dropna(subset=['effective_sell_price', 'contract_price_eur'])
    .groupby('category')['effective_margin_pct'].mean().round(1)
    .sort_values(ascending=False)
)
print('Effective margin % by category (after promo discounts + contract prices):')
print(cat_margin.to_string())

**What to look for:** Compare these margins to the raw `(sell_price - unit_cost)` margins.
Promo discounts and supplier price increases both compress margins. HBC should still lead;
Produce should still be lowest.

In [ ]:
# Reorder alert table using contract prices
reorder_alerts = (
    df_enriched[
        (df_enriched['stock_level'] < df_enriched['reorder_threshold']) &
        (df_enriched['supplier_lead_days'] > 0) &
        (df_enriched['stock_level'] >= 0)
    ]
    [['sku', 'product_name', 'category', 'store_id', 'week',
      'stock_level', 'reorder_threshold', 'supplier_lead_days',
      'supplier_name', 'contract_price_eur', 'min_order_qty']]
    .copy()
)
reorder_alerts['units_to_order'] = (
    (reorder_alerts['reorder_threshold'] - reorder_alerts['stock_level']) * 2
).clip(lower=1)
reorder_alerts['reorder_cost_eur'] = (
    reorder_alerts['units_to_order'] * reorder_alerts['contract_price_eur']
).round(2)
reorder_alerts = reorder_alerts.sort_values('reorder_cost_eur', ascending=False)

print(f'Reorder alerts: {len(reorder_alerts)} rows')
print(f'Total reorder cost: €{reorder_alerts["reorder_cost_eur"].sum():,.2f}')
reorder_alerts.to_excel('reorder_alerts.xlsx', index=False)
print('✅ reorder_alerts.xlsx written')
reorder_alerts.head(10)

---
## Stage 3 — Events as a Demand Signal

In [ ]:
# Load events (fallback file — replace with live crawl output if available)
with open(DATA_DIR / 'local_events_fallback.json', 'r', encoding='utf-8') as f:
    events = json.load(f)

print(f'Loaded {len(events)} events')
for e in events:
    print(f"  {e['date']}  {e['name']:40s}  attendance={e['expected_attendance']:,}  boosts={e['category_boosts']}")

**What is this model for?**

Local events drive predictable surges in demand for specific grocery categories.
The `boost_events()` function below calculates how much more you should order for
affected products — before the event week arrives.

**When to run it:** Once per week, before placing standing orders with suppliers.
Check the event calendar for the next 30 days and apply boost multipliers to reorder quantities.

In [ ]:
# boost_events() — demand boost model
def boost_events(events: list, df: pd.DataFrame) -> pd.DataFrame:
    latest_week = df['week'].max()
    latest_df = df[df['week'] == latest_week].copy()

    def multiplier(attendance: int) -> float:
        if attendance >= 50000: return 2.5
        if attendance >= 25000: return 2.0
        if attendance >= 10000: return 1.5
        return 1.0

    results = []
    for event in events:
        m = multiplier(event['expected_attendance'])
        if m == 1.0:
            continue
        affected = latest_df[latest_df['category'].isin(event['category_boosts'])].copy()
        base = (affected['reorder_threshold'] - affected['stock_level']).clip(lower=0)
        affected = affected.assign(
            event_name=event['name'],
            event_date=event['date'],
            attendance=event['expected_attendance'],
            multiplier=m,
            base_reorder_qty=base,
            boosted_reorder_qty=(base * m).round().astype(int)
        )
        results.append(affected)

    if not results:
        print('No qualifying events found.')
        return pd.DataFrame()

    return (
        pd.concat(results, ignore_index=True)
        [['sku', 'product_name', 'category', 'store_id', 'event_name',
          'event_date', 'attendance', 'multiplier', 'stock_level',
          'base_reorder_qty', 'boosted_reorder_qty']]
        .sort_values('boosted_reorder_qty', ascending=False)
        .reset_index(drop=True)
    )


boosted = boost_events(events, df_enriched)
print(f'Boosted reorder recommendations: {len(boosted)} rows')
boosted.head(15)

---
## Stage 4 — SQLite Database & Financial Summary

In [ ]:
# Create inventory.db with 3 tables
DB_PATH = 'inventory.db'

conn = sqlite3.connect(DB_PATH)

df_enriched.to_sql('inventory',       conn, if_exists='replace', index=False)
supplier_df.to_sql('supplier_prices', conn, if_exists='replace', index=False)
promo_df.to_sql('promo_calendar',     conn, if_exists='replace', index=False)

# Verify row counts
for table in ['inventory', 'supplier_prices', 'promo_calendar']:
    count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn).iloc[0, 0]
    print(f'  {table}: {count:,} rows')

conn.close()
print(f'\n✅ {DB_PATH} created with 3 tables')

**What to look for:** 3 tables loaded cleanly. The database file `inventory.db` now contains
everything that was previously spread across 3 separate files — queryable with standard SQL.

In [ ]:
# SQL Query: supplier price mismatch — which suppliers raised prices?
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT
        s.sku,
        s.product_name,
        s.supplier_name,
        ROUND(s.contract_price_eur, 2)   AS contract_price_eur,
        ROUND(s.inventory_cost_eur, 2)   AS inventory_cost_eur,
        ROUND(s.contract_price_eur - s.inventory_cost_eur, 2) AS gap_eur,
        ROUND((s.contract_price_eur - s.inventory_cost_eur)
              / s.inventory_cost_eur * 100, 1) AS gap_pct
    FROM supplier_prices s
    WHERE s.price_mismatch = 1
    ORDER BY gap_eur DESC
"""

price_gaps = pd.read_sql(query, conn)
conn.close()

print(f'SKUs with supplier price increases: {len(price_gaps)}')
price_gaps

**What to look for:** These are plain-English business questions turned into SQL by Bob.
The gap columns show exactly how much each supplier has raised prices above the system record.
The `gap_pct` column is the most actionable — sort by this to prioritise contract renegotiations.

In [ ]:
# SQL Query: promo effectiveness — which promo types drive the most units?
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT
        p.promo_type,
        COUNT(*)                              AS promo_weeks,
        ROUND(AVG(i.units_sold), 1)           AS avg_units_on_promo,
        ROUND(AVG(p.discount_pct), 1)         AS avg_discount_pct
    FROM promo_calendar p
    JOIN inventory i ON p.sku = i.sku AND p.week = i.week
    WHERE p.discount_pct > 0
    GROUP BY p.promo_type
    ORDER BY avg_units_on_promo DESC
"""

promo_eff = pd.read_sql(query, conn)
conn.close()

print('Promo effectiveness by type:')
promo_eff

**What to look for:** Which promo type generates the most units sold on average?
Compare `avg_discount_pct` against `avg_units_on_promo` — a high discount with low
units sold indicates a poorly targeted promotion.

In [ ]:
# KPI calculations (using contract prices and effective sell prices)
latest_week = df_enriched['week'].max()
latest = df_enriched[df_enriched['week'] == latest_week].copy()
latest_clean = latest.dropna(subset=['sell_price_eur', 'contract_price_eur'])

latest_clean = latest_clean.copy()
latest_clean['effective_sell_price'] = latest_clean['sell_price_eur'] * (
    1 - latest_clean['discount_pct'].fillna(0) / 100
)

total_inv_value  = (latest_clean['stock_level'] * latest_clean['contract_price_eur']).sum()
avg_margin_pct   = (
    (latest_clean['effective_sell_price'] - latest_clean['contract_price_eur'])
    / latest_clean['effective_sell_price'] * 100
).mean()

at_risk = latest[(latest['stock_level'] < latest['reorder_threshold']) & (latest['stock_level'] >= 0)]
n_at_risk = len(at_risk)

zero_stock = latest_clean[latest_clean['stock_level'] == 0].copy()
avg_weekly = df_enriched.groupby('sku')['units_sold'].mean()
zero_stock['avg_weekly_units'] = zero_stock['sku'].map(avg_weekly)
revenue_at_risk = (zero_stock['avg_weekly_units'] * zero_stock['sell_price_eur']).sum()

supplier_exposure = (
    reorder_alerts.merge(mismatches[['sku', 'gap_eur']], on='sku', how='inner')
    .assign(exposure=lambda x: x['units_to_order'] * x['gap_eur'])
    ['exposure'].sum()
)

print('=' * 55)
print('KPI DASHBOARD')
print('=' * 55)
print(f'Total inventory value (contract prices):  €{total_inv_value:,.2f}')
print(f'Avg effective gross margin %:              {avg_margin_pct:.1f}%')
print(f'SKUs at stockout risk (latest week):       {n_at_risk}')
print(f'Estimated revenue at risk:                €{revenue_at_risk:,.2f}')
print(f'Supplier price gap exposure (reorders):   €{supplier_exposure:,.2f}')

In [ ]:
# Export financial_summary.xlsx — 4 tabs
kpi_df = pd.DataFrame({
    'KPI': [
        'Total Inventory Value (€)',
        'Avg Effective Gross Margin %',
        'SKUs at Stockout Risk',
        'Estimated Revenue at Risk (€)',
        'Supplier Price Gap Exposure (€)'
    ],
    'Value': [
        round(total_inv_value, 2),
        round(avg_margin_pct, 1),
        n_at_risk,
        round(revenue_at_risk, 2),
        round(supplier_exposure, 2)
    ]
})

margin_analysis = (
    latest_clean
    .groupby('category')
    .agg(
        avg_effective_margin_pct=('effective_margin_pct', 'mean'),
        total_units_sold=('units_sold', 'sum')
    )
    .round(2)
    .sort_values('avg_effective_margin_pct', ascending=False)
    .reset_index()
)

with pd.ExcelWriter('financial_summary.xlsx', engine='openpyxl') as writer:
    kpi_df.to_excel(writer,         sheet_name='KPI Dashboard',          index=False)
    reorder_alerts.to_excel(writer, sheet_name='Reorder Cost Projection', index=False)
    margin_analysis.to_excel(writer,sheet_name='Margin Analysis',         index=False)
    mismatches.to_excel(writer,     sheet_name='Supplier Price Gaps',     index=False)

print('✅ financial_summary.xlsx written (4 tabs)')

**What to look for:** Four tabs in the workbook.
- **KPI Dashboard:** now includes Supplier Price Gap Exposure — the hidden cost of stale prices
- **Reorder Cost Projection:** uses contract prices, not stale inventory costs
- **Margin Analysis:** reflects actual promo discounts
- **Supplier Price Gaps:** the action list for the buying team